# Language Detection Model

This project focuses on Language Detection, which is a multi-class classification task in the field of NLP. Our goal is to train a model to automatically identify/predict the language of a given text.

While identifying a language might seems obvious to us human beings, it is a complex challenge for a computer. The difficulty lies in the fact that many languages share the exact same alphabet. For example, English, French, and Spanish all use Latin letters, while several Middle Eastern and Asian languages might use similar-looking characters. A machine cannot "read" like we do. instead, it must uncover hidden patterns and statistical fingerprints, such as the frequency of specific character combinations (n-grams) or unique word structures that define each language.

We are using a [Kaggle dataset that contains 10,000 sentences across 17 different languages](https://www.kaggle.com/datasets/basilb2s/language-detection), including English, Arabic, Spanish, and others. To evaluate our success, we will use the Macro-Average F1 score, which ensures that the model is accurate across all languages, regardless of how many sentences they have in the dataset.

## Project Members

| Name | Last 4 Digits of ID |
| :--- | :--- |
| Almog S | 5890 |
| Eliad B | 6279 |

## Import Dependencies

In [26]:
import pandas as pd
import numpy as np
import sys
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score

text_col = 'Text'
label_col = 'Language'

def load_dataset(file_path):
    return pd.read_csv(file_path, encoding='utf-8')

def view_features(raw, vectorized_data, row_idx):
    row = vectorized_data[row_idx]
    nonzero_indices = row.indices # if we wont filter the data, we'll see most of the columns are filled with 0.0
    tfidf_values = row.data
    
    all_feature_names = vectorizer.get_feature_names_out()
    nonzero_feature_names = [all_feature_names[i] for i in nonzero_indices]
    table = pd.DataFrame([tfidf_values], columns=nonzero_feature_names)
    display(table)
    
def predict_language(input_text):
    vector = vectorizer.transform([input_text])
    prediction = knn.predict(vector)
    return prediction[0]

## Loading the dataset

Our dataset is stored in a CSV file that contains 2 columns: the **text**s and its corresponding **language** label. A DataFrame is the perfect tool for organizing the data and the labels into a clean, tabular structure that is easy to manipulate.

In [27]:
df = load_dataset('dataset.csv')
print(f"{df.shape[0]} sentences have been processed.")
df.head()

10337 sentences have been processed.


,Text,Language
0,"Nature, in the broadest sense, is the natural...",English
1,"""Nature"" can refer to the phenomena of the phy...",English
2,"The study of nature is a large, if not the onl...",English
3,"Although humans are part of nature, human acti...",English
4,[1] The word nature is borrowed from the Old F...,English


## Data Splitting

First, we split the data into train set and test set. We chose to split the data into 80% training and 20% testing.

In [28]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(df[text_col], df[label_col], test_size=0.2)#, random_state=42)

train_preview = pd.concat([X_train_raw, y_train], axis=1)
train_preview.columns = ['Text Sample', 'Language']

test_preview = pd.concat([X_test_raw, y_test], axis=1)
test_preview.columns = ['Text Sample', 'Language']

print("### Table 1: Train Set Snapshot (First 5 Rows)")
display(train_preview.head())
print("### Table 2: Test Set Snapshot (First 5 Rows)")
display(test_preview.head())

### Table 1: Train Set Snapshot (First 5 Rows)


,Text Sample,Language
2890,"Por favor, procure por Machine learning na Wik...",Portugeese
3684,de la Wikimedia Foundation.,French
7477,"educato, colto e sofisticato.",Italian
1251,"as far as i'm concerned, this is the best rest...",English
2736,"Embora em geral elogiando o artigo, considerou...",Portugeese


### Table 2: Test Set Snapshot (First 5 Rows)


,Text Sample,Language
925,[63] Inductive logic programming (ILP) is an a...,English
3710,Elle se distingue notamment par l'obligation p...,French
2521,[8] A Wikipédia é um projeto de enciclopédia m...,Portugeese
8963,ويكيبيديا هي موسوعة يمكن لأي مستخدم تعديل وتحر...,Arabic
4603,Ik neem het.,Dutch


## Feature Engineering

In this step, we transform the raw text into a numerical format that the machine learning algorithm can understand. We use the **TF-IDF Vectorizer with Character N-grams** (ranging from 1 to 3 characters): Instead of looking at complete words, the model breaks down each sentence into small sequences of single letters, pairs, and triplets. This is very effective for language detection because every language has a unique statistical "fingerprint" based on how often specific character combinations appear. This method allows the model to differentiate between languages that share the same alphabet and stay accurate even when there are typos or unfamiliar words.

**How does TF-IDF work?**

The value in each cell is calculated using this formula:

$$Score = TF \times IDF$$

Where:

$$TF = \frac{\text{Number of times the sequence appears in the row}}{\text{Total number of sequences in that row}}$$

$$IDF = \log \left( \frac{\text{Total number of rows in the DataFrame}}{\text{Number of rows containing the sequence}} \right)$$

In other words, $TF$ indicates how many times a specific letter sequence appears in a single sentence, while $IDF$ indicates how rare this sequence is across the entire dataset.

**The higher the score, the more important that character sequence is for identifying the language.**

In [30]:
# analyzer='char' tells the model to look at characters, not words.
# ngram_range=(1, 3) looks at single letters, pairs, and triplets.
# norm='l2' will make our data normalized
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(1, 3), norm='l2')
X_train = vectorizer.fit_transform(X_train_raw)
X_test = vectorizer.transform(X_test_raw)

print(X_train.shape)
print(X_test.shape)

(8269, 62701)
(2068, 62701)


Below are examples of the non-zero features of 2 sentences.

In [31]:
row_idx = 0

print(f"Sentence - '{X_train_raw.iloc[row_idx]}'")
view_features(X_train_raw, X_train, row_idx)

print("=" * 50 + "\n");

print(f"Sentence - '{X_test_raw.iloc[row_idx]}'")
view_features(X_test_raw, X_test, row_idx)

Sentence - 'Por favor, procure por Machine learning na Wikipédia para buscar por títulos alternativos.'


,p,o,r,,f,a,v,",",c,u,...,lte,ter,ern,rna,nat,ati,tiv,ivo,vos,os.
0,0.159489,0.160041,0.224027,0.191976,0.030151,0.219668,0.057491,0.027399,0.07787,0.072775,...,0.086794,0.04802,0.069522,0.076249,0.065307,0.049562,0.065397,0.084074,0.097346,0.087147



Sentence - '[63] Inductive logic programming (ILP) is an approach to rule-learning using logic programming as a uniform representation for input examples, background knowledge, and hypotheses.'


,,(,(i,a,a,an,ap,as,b,ba,...,ve,w,wl,wle,x,xa,xam,y,yp,ypo
0,0.207079,0.03507,0.059499,0.07993,0.027202,0.052303,0.037846,0.035737,0.021769,0.036679,...,0.033834,0.020198,0.058277,0.06018,0.028927,0.046154,0.050944,0.020464,0.052582,0.067444


## Model Training (KNN)

Now that our text is represented as a matrix of numerical vectors (TF-IDF), we can apply the **KNN algorithm**. 

KNN predicts the language of a new sentence by looking at the $k$ most similar sentences in our training data.

The logic of our algorithm implementation is structured as follows:

- We use the **cosine similarity score** to check how similar a test sentence is to the sentences in our training set. The score ranges from [-1,1] and is calculated using the dot product and magnitude:

$$\displaystyle{\text{Cosine similarity} = \cos \left( \theta \right) = \frac{ A \cdot B }{ \left\| A \right\| \cdot \left\| B \right\| } }$$

In this formula, $A$ represents a row vector from $X_{\text{test}}$ and $B$ represents a row vector from $X_{\text{train}}^T$.

Since our data is already L2 normalized by the TF-IDF vectorizer, the magnitude of each vector is 1 ($\|A\| = 1$ and $\|B\| = 1$). This simplifies the formula:

$$\displaystyle{ \text{Cosine similarity} = \frac{ A \cdot B }{ 1 \cdot 1 } = A \cdot B }$$
This means we only need to calculate the **dot product** between the test row and all training rows (using $X_{\text{test}} \cdot X_{\text{train}}^T$). 

- We sort the similarity scores and select the $k$ closest neighbors.
- We check the labels (languages) of these $k$ neighbors. The language that appears most frequently among them is chosen as the final prediction.

**Note**: We will explore different K-values using Cross-Validation at the [Evaluation](#Evaluation) step to find the optimal hyperparameters.

In [44]:
class KNN:
    def __init__(self, k):
        self.k = k
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        self.X_train = X
        self.y_train = np.array(y)

    def predict(self, X_test):
        similarities = X_test.dot(self.X_train.T).toarray() # dot product
        predictions = []
        
        for i in range(similarities.shape[0]):
            neighbors_indexes = np.argsort(similarities[i])[-self.k:] # get k indexes of the best values (nearest neighbors)
            k_neighbor_labels = self.y_train[neighbors_indexes] # get their labels

            unique_labels, counts = np.unique(k_neighbor_labels, return_counts=True) # count how much times each label appears
            best_label_index = np.argmax(counts) # index of label that appears the most
            best_label = unique_labels[best_label_index] # retreive the label
            predictions.append(best_label)
            
        return np.array(predictions)

knn = KNN(k=11)
knn.fit(X_train, y_train)

## Model Testing

Now, we want to check if our model is actually successful. It is important to know if the model truly "learned" how to identify languages or if the results were just a matter of luck. To do this, we will test the model on the Test Set data that the model has never seen before.

In [45]:
y_pred = knn.predict(X_test)
results_table = pd.DataFrame({'Text': X_test_raw[:20], 'Actual': y_test[:20], 'Predicted': y_pred[:20]})
display(results_table)

,Text,Actual,Predicted
925,[63] Inductive logic programming (ILP) is an a...,English,English
3710,Elle se distingue notamment par l'obligation p...,French,French
2521,[8] A Wikipédia é um projeto de enciclopédia m...,Portugeese,Portugeese
8963,ويكيبيديا هي موسوعة يمكن لأي مستخدم تعديل وتحر...,Arabic,Arabic
4603,Ik neem het.,Dutch,Dutch
6646,"только подумайте о той бедной матери-нарциссе,...",Russian,Russian
6607,"ура, я очень ценю это.",Russian,Russian
70,Weather can have both beneficial and harmful e...,English,English
3867,des données utilisées pour l’entrainement mais...,French,French
2747,[21] Editores de obras de referência tradicion...,Portugeese,Portugeese


Great! our predictions match the results.

## Evaluation

This step will help us answer two key questions:

- How accurate is the model on new text, one that it has never seen before?

- How often does it make mistakes?

We will use the Accuracy score and the Macro F1-score to see the full picture and ensure the model performs well across all 17 languages.

In [46]:
print(f"Overall Accuracy: {accuracy_score(y_test, y_pred):.2%}")
print("\nFull Performance Report:")
print(classification_report(y_test, y_pred))

Overall Accuracy: 98.21%

Full Performance Report:
              precision    recall  f1-score   support

      Arabic       1.00      1.00      1.00       113
      Danish       1.00      0.91      0.95        98
       Dutch       0.95      0.97      0.96       114
     English       0.96      0.99      0.98       270
      French       0.98      1.00      0.99       202
      German       0.98      0.99      0.98        91
       Greek       1.00      1.00      1.00        70
       Hindi       1.00      1.00      1.00        10
     Italian       0.96      0.98      0.97       134
     Kannada       1.00      1.00      1.00        60
   Malayalam       1.00      1.00      1.00       113
  Portugeese       0.99      0.96      0.98       164
     Russian       1.00      1.00      1.00       133
     Spanish       0.98      0.98      0.98       164
    Sweedish       0.98      0.97      0.97       140
       Tamil       1.00      1.00      1.00       111
     Turkish       0.99      0

## DIY

Enter your desired text and check our model!

In [41]:
text = 'Input' # EDIT THIS VALUE TO PREDICT YOUR TEXT'S LANGUAGE
print(f"Input: '{text}' -> Predicted: {predict_language(text)}")

Input: 'Input' -> Predicted: English
